# Cambrian-P: compare the models' results

Reads the results that `cambrian_p_colab.ipynb` saved to Google Drive and draws charts comparing the models. It needs **no GPU and no install**: a free CPU runtime is enough, and it takes under a minute.

Run the cells in order. Every chart is shown here and saved as a PNG in `GRAPHS_DIR` on your Drive.

**Charts**
1. Overall score per model
2. Score per model for each video
3. Every question: which models got it right, and what each answered
4. How often each pair of models gives the same answer
5. Open-ended answers: how long each model's description is (flags empty and runaway answers)
6. Predicted camera paths, one panel per model and video

Scores follow VSI-Bench (`lmms-eval/lmms_eval/tasks/vsibench/utils.py`), as in the main notebook: multiple choice and one-word answers by exact match on the model's first word, numbers by mean relative accuracy. Only questions with a true answer are scored.

In [ ]:
# @title 1. Settings and Google Drive
RESULTS_DIR = "/content/drive/MyDrive/Cambrian-P/results"  # @param {type:"string"}
GRAPHS_DIR = "/content/drive/MyDrive/Cambrian-P/results/graphs"  # @param {type:"string"}

# Models in the order they appear in every chart (each model's frame count is shown with its name).
MODELS = {
    "Cambrian-P-7B": 128,
    "Cambrian-P-7B-32f": 32,
    "Cambrian-P-7B-Mix-MA": 64,
    "Cambrian-P-7B-Mix-3R": 32,
    "Cambrian-P-7B-Mix-CamS": 128,
}

import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass  # not in Colab: RESULTS_DIR must be a local folder
os.makedirs(GRAPHS_DIR, exist_ok=True)
found = [m for m in MODELS if os.path.isdir(os.path.join(RESULTS_DIR, m))]
print("Model folders found:", ", ".join(found) or "none")
missing = [m for m in MODELS if m not in found]
if missing:
    print("Not found (skipped):", ", ".join(missing))

In [ ]:
# @title 2. Load and score the answers
import csv, glob, json, re
import numpy as np
import pandas as pd

def first_word(answer):
    return answer.split(" ")[0].rstrip(".").strip()

def as_float(text):
    try:
        return float(text)
    except ValueError:
        return None

def score(answer, truth):
    """VSI-Bench scoring: 1/0 exact match, 0-1 mean relative accuracy for numbers, NaN without a true answer."""
    truth = str(truth).strip()
    if not truth:
        return np.nan
    if as_float(truth) is not None:
        target, pred = as_float(truth), as_float(first_word(answer))
        if pred is None:
            return 0.0
        if target == 0:
            return float(pred == 0)
        return float(np.mean(abs(pred - target) / abs(target) <= 1 - np.linspace(0.5, 0.95, 10)))
    if len(truth.split()) == 1:
        return float(first_word(answer).lower() == truth.lower())
    return float(answer.strip().rstrip(".").lower() == truth.rstrip(".").lower())

FRAMING = ("These are frames of a video.", "Please answer the question", "Answer with the option", "Options:")

def question_text(question):
    """The question sentence alone, without the VSI-Bench framing or the options."""
    lines = [l for l in question.splitlines() if l.strip() and not l.startswith(FRAMING) and not re.match(r"^[A-H]\. ", l)]
    return " ".join(lines)

def load_model(model):
    """Rows from the model's results.csv, or from per-video answers.json if a run stopped before writing it."""
    folder = os.path.join(RESULTS_DIR, model)
    path = os.path.join(folder, "results.csv")
    if os.path.exists(path):
        with open(path, newline="") as f:
            return list(csv.DictReader(f))
    rows = []
    for p in sorted(glob.glob(os.path.join(folder, "*", "answers.json"))):
        rows += json.load(open(p))
    return rows

rows = []
for model in found:
    for r in load_model(model):
        if r.get("model_answer", "").startswith("ERROR:"):
            continue
        rows.append(dict(r, model=model))
df = pd.DataFrame(rows)
df["model"] = pd.Categorical(df["model"], categories=found, ordered=True)
df["score"] = [score(a, t) for a, t in zip(df["model_answer"], df["true_answer"])]
df["q"] = df["question"].map(question_text)
# Keep questions in the order they were asked, per video.
order = df.drop_duplicates(["video", "question"])[["video", "question"]].reset_index(drop=True)
order["qid"] = order.groupby("video").cumcount() + 1
df = df.merge(order, on=["video", "question"])
videos = list(dict.fromkeys(df["video"]))
scored = df.dropna(subset=["score"])
print(f"{len(found)} models, {len(videos)} videos ({', '.join(videos)}), "
      f"{df[['video', 'question']].drop_duplicates().shape[0]} questions, "
      f"{scored[['video', 'question']].drop_duplicates().shape[0]} with a true answer")

In [ ]:
# @title Chart style
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import Image, display

SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE = "#2a78d6"
GOOD, PARTIAL, BAD, EMPTY = "#0ca30c", "#ec835a", "#d03b3b", "#f0efec"
BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.edgecolor": GRID, "axes.grid": False, "font.size": 10, "axes.titlesize": 12,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False,
})

def label(model):
    return f"{model.replace('Cambrian-P-', '')} ({MODELS.get(model, '?')}f)"

def save(fig, name):
    path = os.path.join(GRAPHS_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    display(Image(path))
    print("Saved", path)

def ink_on(rgba):
    """Dark or light text, whichever reads better on this cell color."""
    r, g, b = rgba[:3]
    return INK if 0.2126 * r + 0.7152 * g + 0.0722 * b > 0.55 else "white"

In [ ]:
# @title Chart 1: overall score per model
summary = scored.groupby("model", observed=False)["score"].agg(["mean", "sum", "count"]).reindex(found)
fig, ax = plt.subplots(figsize=(8, 0.55 * len(found) + 1.2))
y = np.arange(len(found))[::-1]
ax.barh(y, 100 * summary["mean"], height=0.55, color=BLUE)
for yi, (m, r) in zip(y, summary.iterrows()):
    ax.text(100 * r["mean"] + 1.5, yi, f"{100 * r['mean']:.0f}%  ({r['sum']:g}/{r['count']:.0f})", va="center", color=INK)
ax.set_yticks(y, [label(m) for m in found])
ax.set_xlim(0, 115)
ax.set_xticks([0, 25, 50, 75, 100], ["0%", "25%", "50%", "75%", "100%"])
ax.xaxis.grid(True, color=GRID, lw=0.8)
ax.set_axisbelow(True)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.set_title("Score per model, all videos")
ax.set_xlabel("Questions answered correctly (questions with a true answer)")
save(fig, "1_score_per_model.png")

In [ ]:
# @title Chart 2: score per model for each video
per_video = scored.pivot_table(index="model", columns="video", values="score", aggfunc="mean", observed=False).reindex(found)
counts = scored.groupby("video")["question"].nunique()
fig, ax = plt.subplots(figsize=(1.6 * len(videos) + 3, 0.6 * len(found) + 1.5))
im = ax.imshow(100 * per_video.values, cmap=BLUES, vmin=0, vmax=100, aspect="auto")
for i in range(per_video.shape[0]):
    for j in range(per_video.shape[1]):
        v = per_video.values[i, j]
        if np.isnan(v):
            ax.text(j, i, "no answers", ha="center", va="center", color=INK_2, fontsize=8)
        else:
            ax.text(j, i, f"{100 * v:.0f}%", ha="center", va="center", color=ink_on(im.cmap(im.norm(100 * v))), fontweight="bold")
ax.set_xticks(range(len(per_video.columns)), [f"{v}\n({counts.get(v, 0)} scored)" for v in per_video.columns])
ax.set_yticks(range(len(found)), [label(m) for m in found])
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title("Score per model and video")
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, format="%d%%").outline.set_visible(False)
save(fig, "2_score_per_video.png")

In [ ]:
# @title Chart 3: every question, every model
# One block per video. Each cell shows the model's answer: green ✓ right, red ✗ wrong, orange ~ partly right (numbers).
# Questions without a true answer are left out here; see chart 5 and the CSVs for those.
import textwrap
for video in videos:
    sub = scored[scored["video"] == video]
    if sub.empty:
        continue
    qs = sub.drop_duplicates("qid").sort_values("qid")
    n_q, n_m = len(qs), len(found)
    fig, ax = plt.subplots(figsize=(2.2 + 1.35 * (n_m + 1) + 5.5, 0.62 * n_q + 1.4))
    ax.set_xlim(-0.5, n_m + 0.5 + 0.9)
    ax.set_ylim(n_q - 0.5, -0.5)
    for i, (_, q) in enumerate(qs.iterrows()):
        ax.add_patch(plt.Rectangle((-0.48, i - 0.46), 0.96, 0.92, color=EMPTY, lw=0))
        ax.text(0, i, q["true_answer"], ha="center", va="center", fontweight="bold")
        right = 0
        for j, m in enumerate(found, start=1):
            hit = sub[(sub["qid"] == q["qid"]) & (sub["model"] == m)]
            if hit.empty:
                ax.add_patch(plt.Rectangle((j - 0.48, i - 0.46), 0.96, 0.92, color=EMPTY, lw=0))
                ax.text(j, i, "—", ha="center", va="center", color=INK_2)
                continue
            s, ans = hit["score"].iloc[0], hit["model_answer"].iloc[0].strip()
            color, mark = (GOOD, "✓") if s == 1 else (BAD, "✗") if s == 0 else (PARTIAL, "~")
            right += s
            ax.add_patch(plt.Rectangle((j - 0.48, i - 0.46), 0.96, 0.92, color=color, lw=0))
            shown = (first_word(ans) or "(empty)")[:10]
            ax.text(j, i, f"{mark} {shown}", ha="center", va="center", color="white", fontweight="bold")
        ax.text(n_m + 0.62, i, f"{right:g}/{n_m}", ha="left", va="center", color=INK_2)
    ax.set_xticks(range(n_m + 1), ["True\nanswer"] + [label(m).replace(" (", "\n(") for m in found])
    ax.xaxis.tick_top()
    ax.set_yticks(range(n_q), [textwrap.shorten(f"Q{q['qid']}. {q['q']}", 70, placeholder="…") for _, q in qs.iterrows()])
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(f"{video}: answers to each question (right column: models correct)", pad=34)
    save(fig, f"3_questions_{os.path.splitext(video)[0]}.png")

In [ ]:
# @title Chart 4: how often two models give the same answer
# Over all questions with a true answer: the share where both models' answers (first word) match.
first = scored.assign(a=scored["model_answer"].map(lambda a: first_word(a).lower())) \
              .pivot_table(index=["video", "qid"], columns="model", values="a", aggfunc="first", observed=False)
cols = [m for m in found if m in first.columns]
agree = pd.DataFrame(np.nan, index=cols, columns=cols)
for a in cols:
    for b in cols:
        both = first[a].notna() & first[b].notna()
        agree.loc[a, b] = (first.loc[both, a] == first.loc[both, b]).mean() if both.any() else np.nan
fig, ax = plt.subplots(figsize=(0.95 * len(agree) + 3.2, 0.8 * len(agree) + 1.2))
im = ax.imshow(100 * agree.values.astype(float), cmap=BLUES, vmin=0, vmax=100)
for i in range(len(agree)):
    for j in range(len(agree)):
        v = agree.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{100 * v:.0f}%", ha="center", va="center", color=ink_on(im.cmap(im.norm(100 * v))))
names = [label(m) for m in agree.columns]
ax.set_xticks(range(len(names)), names, rotation=30, ha="right")
ax.set_yticks(range(len(names)), names)
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title("How often two models give the same answer")
save(fig, "4_model_agreement.png")

In [ ]:
# @title Chart 5: open-ended answers (no true answer): length per model and video
# Long answers that repeat themselves usually mean the model got stuck in a loop.
def words(a):
    return len(a.split())

def looping(a):
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", a) if s.strip()]
    return len(sents) >= 6 and len(set(sents)) <= len(sents) / 2

open_q = df[df["true_answer"].astype(str).str.strip() == ""]
open_q = open_q[~open_q["question"].str.contains("single word or phrase")]  # number questions without a true answer
if open_q.empty:
    print("No open-ended questions found.")
else:
    for qtext, sub in open_q.groupby("q", sort=False):
        n = sub.pivot_table(index="model", columns="video", values="model_answer", aggfunc="first", observed=False).reindex(found)
        counts = n.map(lambda a: words(a) if isinstance(a, str) else np.nan)
        fig, ax = plt.subplots(figsize=(1.6 * n.shape[1] + 3, 0.6 * len(found) + 1.6))
        im = ax.imshow(counts.values.astype(float), cmap=BLUES, vmin=0, vmax=max(150, np.nanmax(counts.values) if counts.notna().any().any() else 150), aspect="auto")
        for i in range(n.shape[0]):
            for j in range(n.shape[1]):
                a = n.values[i, j]
                if not isinstance(a, str):
                    ax.text(j, i, "—", ha="center", va="center", color=INK_2)
                elif not a.strip():
                    ax.text(j, i, "empty", ha="center", va="center", color=BAD, fontweight="bold")
                else:
                    txt = f"{words(a)} words" + ("\nloops" if looping(a) else "")
                    ax.text(j, i, txt, ha="center", va="center", color=ink_on(im.cmap(im.norm(words(a)))), fontsize=9)
        ax.set_xticks(range(n.shape[1]), n.columns)
        ax.set_yticks(range(len(found)), [label(m) for m in found])
        ax.tick_params(length=0)
        for s in ax.spines.values():
            s.set_visible(False)
        ax.set_title(f"Answer length: “{textwrap.shorten(qtext, 60, placeholder='…')}”")
        safe = re.sub(r"[^A-Za-z0-9]+", "_", qtext)[:40].strip("_")
        save(fig, f"5_open_answer_length_{safe}.png")

In [ ]:
# @title Chart 6: predicted camera paths (bird's-eye view)
# One row per video, one column per model. Each panel has its own scale: the models' units are relative
# and not comparable, so compare the shapes (turns, loops, where it ends), not the sizes.
poses = {(v, m): os.path.join(RESULTS_DIR, m, os.path.splitext(v)[0], "poses.npz") for v in videos for m in found}
if not any(os.path.exists(p) for p in poses.values()):
    print("No poses.npz files found, so no camera paths to plot.")
else:
    fig, axes = plt.subplots(len(videos), len(found), figsize=(2.6 * len(found), 2.6 * len(videos)), squeeze=False)
    for i, v in enumerate(videos):
        for j, m in enumerate(found):
            ax = axes[i, j]
            ax.set_xticks([])
            ax.set_yticks([])
            for s in ax.spines.values():
                s.set_color(GRID)
                s.set_visible(True)
            if i == 0:
                ax.set_title(label(m), fontsize=9, loc="center")
            if j == 0:
                ax.set_ylabel(v, fontsize=9, color=INK)
            if not os.path.exists(poses[v, m]):
                ax.text(0.5, 0.5, "no poses", ha="center", va="center", color=INK_2, transform=ax.transAxes)
                continue
            pos = np.load(poses[v, m])["c2w"][:, :3, 3]
            ax.plot(pos[:, 0], pos[:, 2], color=BLUE, lw=2)
            ax.plot(*pos[0, [0, 2]], "o", ms=8, mfc=SURFACE, mec=GOOD, mew=2)
            ax.plot(*pos[-1, [0, 2]], "X", ms=9, color=BAD, mec=SURFACE, mew=1)
            ax.set_aspect("equal", adjustable="datalim")
    fig.suptitle("Predicted camera path from above (○ start, ✕ end; x = right, up = forward from the first frame)",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    save(fig, "6_camera_paths.png")

In [ ]:
# @title Save the scored table
out = df[["video", "qid", "q", "true_answer", "model", "model_answer", "score"]].sort_values(["video", "qid", "model"])
out.rename(columns={"q": "question"}).to_csv(os.path.join(GRAPHS_DIR, "scored_answers.csv"), index=False)
print("Saved", os.path.join(GRAPHS_DIR, "scored_answers.csv"))
print("All charts are in", GRAPHS_DIR)